In [26]:
"This IPYNB file is only used as a test, and a tool for deploying the model. The real Code goes in the same file, but "
"with a .py extension rather than a .ipynb"

'with a .py extension rather than a .ipynb'

In [27]:
from pathlib import Path
import pandas as pd

# Try common working directories (notebook folder and workspace root).
candidate_paths = [
    Path("../../../../SOURCES_AND_DATASHEETS/usgs_data_USGS-01646500.csv"),
    Path("backend/SOURCES_AND_DATASHEETS/usgs_data_USGS-01646500.csv"),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not locate usgs_data_USGS-01646500.csv")
print(f"Using CSV file at: {csv_path}")

df = pd.read_csv(csv_path)
df

Using CSV file at: ..\..\..\..\SOURCES_AND_DATASHEETS\usgs_data_USGS-01646500.csv


,Unnamed: 0,gage_height_ft,streamflow_cfs,dissolved_oxygen_mg_l,pH,specific_conductance_us_cm,temperature_c,turbidity_fnu,precipitation,rain,snowfall,snow_depth,soil_moisture_0_to_1cm,soil_moisture_1_to_3cm,temperature_2m,wind_speed_10m,vapour_pressure_deficit,precip_3hr,precip_24hr,precip_72hr
0,2023-07-06 04:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,23.85,6.287130,0.264494,NaN,NaN,NaN
1,2023-07-06 05:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,23.45,6.130579,0.226730,NaN,NaN,NaN
2,2023-07-06 06:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,24.00,5.154416,0.323828,NaN,NaN,NaN
3,2023-07-06 07:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,22.90,5.351785,0.172635,NaN,NaN,NaN
4,2023-07-06 08:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,22.05,5.411986,0.071846,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176682,2026-07-10 23:50:00+00:00,3.73,7820.0,7.7,8.6,340.0,28.3,35.7,0.0,0.0,0.0,0.0,NaN,NaN,25.90,3.017350,0.437354,4.5,20.4,457.500001
176683,2026-07-11 00:00:00+00:00,NaN,NaN,7.7,8.6,340.0,28.3,35.7,0.0,0.0,0.0,0.0,NaN,NaN,25.30,4.445672,0.294220,3.6,20.4,457.200001
176684,2026-07-11 01:00:00+00:00,NaN,NaN,7.7,8.6,340.0,28.3,35.7,0.0,0.0,0.0,0.0,NaN,NaN,24.00,4.965521,0.208287,3.3,20.1,456.900001
176685,2026-07-11 02:00:00+00:00,NaN,NaN,7.7,8.6,340.0,28.3,35.7,0.0,0.0,0.0,0.0,NaN,NaN,23.50,4.500000,0.169943,3.0,19.8,456.600001


In [28]:
# Flood Action Stage: 5 ft
# Minor Flood Stage: 10 ft
# Moderate Flood Stage: 12 ft
# Major Flood Stage: 14 ft
FLOOD_ACTION_STAGE = 5.0
MINOR_FLOOD_STAGE = 10.0
MODERATE_FLOOD_STAGE = 12.0
MAJOR_FLOOD_STAGE = 14.0

In [29]:
#df.hist(figsize=(10, 6))
# get the instances where Gage Height is > 5
df = df.dropna(subset=['gage_height_ft'])
df_flood = df[df['gage_height_ft'] > FLOOD_ACTION_STAGE]
df_minor_flood = df[df['gage_height_ft'] > MINOR_FLOOD_STAGE]
df_moderate_flood = df[df['gage_height_ft'] > MODERATE_FLOOD_STAGE]
df_major_flood = df[df['gage_height_ft'] > MAJOR_FLOOD_STAGE]

# print the lengths of each of the dataframes
print(f"Total records: {len(df)}")
print(f"Flood Action Stage records: {len(df_flood)}")
print(f"Minor flood records: {len(df_minor_flood)}")
print(f"Moderate flood records: {len(df_moderate_flood)}")
print(f"Major flood records: {len(df_major_flood)}")

Total records: 176230
Flood Action Stage records: 9772
Minor flood records: 131
Moderate flood records: 0
Major flood records: 0


In [ ]:
# df should have a datetime column and a binary flag column for action stage
# adjust column names to match your data

# Name the 'Unnamed: 0' column as 'datetime' and convert it to datetime type
df['datetime'] = pd.to_datetime(df['Unnamed: 0'])

df = df.sort_values('datetime').reset_index(drop=True)

# how many hours of gap counts as "the storm ended" (tune this to your data's
# sampling frequency, e.g. 6-12h for hourly gauge data, 24-48h for daily)
GAP_HOURS = 12

# isolate just the flagged (action-stage) rows
flood_rows = df[df['gage_height_ft'] > FLOOD_ACTION_STAGE].copy()

# time since previous flagged reading, saved in column 'gap'
flood_rows['gap'] = flood_rows['datetime'].diff()

# start a new event whenever the gap exceeds threshold (or it's the first row)
flood_rows['new_event'] = (
    flood_rows['gap'].isna() | (flood_rows['gap'] > pd.Timedelta(hours=GAP_HOURS))
)
flood_rows['event_id'] = flood_rows['new_event'].cumsum()

# summarize each event
events = flood_rows.groupby('event_id').agg(
    start=('datetime', 'min'),
    end=('datetime', 'max'),
    n_readings=('datetime', 'count'),
    peak_gage_height=('gage_height_ft', 'max')  # adjust column name as needed
).reset_index(drop=True)

events['duration_hours'] = (events['end'] - events['start']).dt.total_seconds() / 3600

print(f"Total flagged readings: {len(flood_rows)}")
print(f"Independent storm events: {len(events)}")
print(events)


Total flagged readings: 131
Independent storm events: 1
                      start                       end  n_readings  \
0 2025-05-15 09:30:00+00:00 2025-05-16 18:00:00+00:00         131   

   peak_gage_height  duration_hours  
0             11.72            32.5  
